# 03b — Feature Encoding

Converts the engineered dataset into a fully numeric modeling matrix:

- **Frequency encoding** for high-cardinality categoricals (`ProductCD`, `card4`, `card6`, email domains, `DeviceType`, `id_31` browser). Label-free — uses marginal frequencies only, so no target leakage.
- **M1–M9** T/F flags mapped to 1/0 (NaN preserved — informative).
- Saves the canonical **`feature_list.json`** consumed identically by notebooks 04 and 06 (single source of truth, no drift).

Input: `data/processed/time_window_features.csv` → Output: `data/processed/model_features.csv` + `feature_list.json`

In [1]:
# CELL 1 - Load engineered dataset from notebook 03
import os
import json
import numpy as np
import pandas as pd

df = pd.read_csv("../data/processed/time_window_features.csv")
print(f"Loaded: {df.shape}")

/var/folders/03/lp_c5nd95wvd8k80883y1h6w0000gn/T/ipykernel_28839/2849165151.py:7: DtypeWarning: Columns (45,46,47,49,50,51,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/time_window_features.csv")


Loaded: (590540, 77)


In [2]:
# CELL 2 - Frequency-encode high-cardinality categoricals
# Label-free encoding: maps each category to its marginal frequency.
# No target information used -> no target leakage.
cat_cols = ["ProductCD", "card4", "card6",
            "P_emaildomain", "R_emaildomain", "DeviceType", "id_31"]

for c in cat_cols:
    if c in df.columns:
        freq = df[c].value_counts(normalize=True)
        df[f"{c}_freq"] = df[c].map(freq).astype("float32")
        print(f"{c:>15}: {df[c].nunique():>4} categories -> {c}_freq")

      ProductCD:    5 categories -> ProductCD_freq
          card4:    4 categories -> card4_freq
          card6:    4 categories -> card6_freq
  P_emaildomain:   59 categories -> P_emaildomain_freq
  R_emaildomain:   60 categories -> R_emaildomain_freq
     DeviceType:    2 categories -> DeviceType_freq
          id_31:  130 categories -> id_31_freq


In [3]:
# CELL 3 - M1-M9 match flags: T/F -> 1/0 (NaN stays NaN)
for c in [f"M{i}" for i in range(1, 10)]:
    if c in df.columns:
        df[c] = df[c].map({"T": 1, "F": 0}).astype("float32")
print("M columns mapped to 1/0")

M columns mapped to 1/0


In [4]:
# CELL 4 - Build the canonical feature list
# Everything numeric stays; drop identifiers, target, and raw strings
# that now have encoded versions.
exclude = {"TransactionDT", "customer_id", "isFraud"}
exclude |= {c for c in cat_cols if c in df.columns}   # raw strings replaced by _freq

feature_list = [c for c in df.columns if c not in exclude]

# Guard: model matrix must be fully numeric
non_numeric = df[feature_list].select_dtypes(exclude="number").columns.tolist()
assert not non_numeric, f"Still non-numeric: {non_numeric}"

os.makedirs("../data/processed", exist_ok=True)
with open("../data/processed/feature_list.json", "w") as f:
    json.dump(feature_list, f, indent=1)

print(f"{len(feature_list)} model features -> feature_list.json")

74 model features -> feature_list.json


In [5]:
# CELL 5 - Save modeling dataset
df.to_csv("../data/processed/model_features.csv", index=False)
print(f"Saved model_features.csv: {df.shape}")

Saved model_features.csv: (590540, 84)
